In [11]:
# !pip install -U duckduckgo-search

In [12]:
# !pip install openai-agents

### Setup

In [13]:
import os
# from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")
else:
    print("OPENAI_API_KEY is set.")


# client = OpenAI(api_key=OPENAI_API_KEY)

MODEL = "gpt-4.1-mini"

OPENAI_API_KEY is set.


### Step 1: Define the tools

In [14]:
@function_tool
def search_web(query: str):
    """Search the web using duckduckgo browser. Return 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"\u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [15]:
@function_tool
def fetch_url(url: str):
    """Fetching the content from url using trafilatura"""
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"\u2705 fetch_url: Got {len(text)} characters from {url[:60]}")
            return text
    print(f"\u274c Failed to extract text from {url}.")
    return f"Could not extract text from {url}. Try a diffrent source."

### Step 2: The Agents

#### **Research Agent**

In [16]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""


research_agent = Agent(
    name="ResearchAgent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url],
)

#### **Ochestrator Agent**

In [17]:
ORCHESTRATOR_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.
"""

tool_test = research_agent.as_tool(
    tool_name="research_agent",
    tool_description="Research a topic and return a brief with key facts, statistics, themes, and source URLs. Pass the topic as input."
)

ochestrator_agent = Agent(
    name="OchestratorAgent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini",  # o4-mini is used for reasoning 
    tools=[tool_test]
)

### Lets have it!


In [18]:
result = await Runner.run(
    ochestrator_agent,
    input="Research the following topic and produce a comprehensive research brief: Best PHD's in Remote sensing",
    max_turns=30,
)

✅ search_web: Got results for Best PhD programs in Remote Sensing
✅ fetch_url: Got 11868 characters from https://www.geologyprograms.com/rankings/best-remote-sensing
✅ fetch_url: Got 5620 characters from https://edurank.org/engineering/remote-sensing/
✅ fetch_url: Got 13144 characters from https://www.gisdegree.org/programs/remote-sensing-degrees/
✅ search_web: Got results for Top PhD programs in Remote Sensing worldwide
❌ Failed to extract text from https://www.phdportal.com/search/phd/geographical-information-systems.
❌ Failed to extract text from https://www.findaphd.com/phds/remote-sensing/?30M7yyj2.
✅ fetch_url: Got 5620 characters from https://edurank.org/engineering/remote-sensing/
✅ search_web: Got results for best universities for PhD in Remote Sensing research
✅ fetch_url: Got 855 characters from https://elitedegrees.net/remote-sensing/united-states/
✅ search_web: Got results for Remote Sensing PhD scholarships and funding
✅ search_web: Got results for PhD Remote Sensing Mass

Error getting response


APIConnectionError: Connection error.

In [ ]:
print(f"Agent: {result.last_agent.name}")
print(f"----")
display(Markdown(result.final_output))

Agent: OchestratorAgent
----


I’ve selected the second brief as the most comprehensive overview of top global PhD opportunities in remote sensing:

Key Facts and Statistics:
- Remote sensing is a global research field with significant academic output from universities across the USA, China, Europe, and Australia.
- Top universities in remote sensing by research impact and publication volume include Wuhan University (China), Xidian University (China), Technical University of Munich (Germany), University of Maryland, College Park (USA), and California Institute of Technology (USA).
- The USA accounts for eight of the top 50 institutions, while China dominates with 25 universities.
- Other notable universities with strong remote sensing programs and research impact are MIT, University of Colorado Boulder, University of Arizona, University of Washington – Seattle, Stanford University, Delft University of Technology, and University of Wisconsin–Madison.
- Rankings are based on citations, research impact, faculty quality, and international collaboration.

Main Themes and Arguments:
- Leading remote sensing programs are characterized by strong interdisciplinary research facilities, collaboration opportunities, and funding.
- Many top programs offer fully funded PhD positions with state-of-the-art labs and projects addressing climate change, environmental monitoring, agriculture, urban planning, and defense.
- The US, China, and European institutions lead in providing high-quality academic training and research output in remote sensing technologies.
- Opportunities expand from traditional remote sensing to include radar engineering, environmental physics, image analysis, AI integration, and sensor development.

Notable Data Points:
- Wuhan University leads globally with the highest score (262.7) in remote sensing research.
- California Institute of Technology has a highly competitive acceptance rate (~3%) and ranks #10 globally in remote sensing.
- University of Maryland, College Park, ranks #9 and is well-known for remote sensing-related research.
- Funding availability and scholarships are frequently part of PhD offerings listed on academic platforms such as FindAPhD and Scholar Nexus.
- Many top-ranked universities have acceptance rates between 15-40%, but some like Caltech and Stanford are highly selective (<5%).

Top Source URLs:
1. https://edurank.org/engineering/remote-sensing/
2. https://stats.areppim.com/stats/stats_uni2024eng_remote.htm
3. https://scholarnexus.ai/positions/phds/Remote-Sensing?subFields=318&program=0
4. https://www.findaphd.com/phds/remote-sensing/?30M7yyj2